# A4 — EvaLoop: code ↔ pseudocode loop (Colab runner)

Runs the standard EvaLoop loop (baseline) and the new **code ↔ pseudocode** loop (plus the
no-assert ablation) on all 378 MBPP+ tasks with small open models through vLLM, then produces
`summary.csv`, the analysis tables, the manual-inspection sample and the submission folder.

**Runtime:** `Runtime → Change runtime type → GPU`. An **L4** or **A100** is recommended (7–8B models in bf16).
On a **T4**, set `DTYPE = "half"` and use the smaller models listed in the config cell.

Results are written to Google Drive so a disconnected session can resume: finished runs are skipped.
Generated code is executed only inside this Colab VM, through EvalPlus's sandboxed `untrusted_check`
(subprocess with time/memory limits and a reliability guard).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ---------------- configuration ----------------
FORK_URL = "https://github.com/shreyas457/EvaLooop.git"   # your public fork
BRANCH   = "a4-pseudocode-loop"

MODELS = [
    "Qwen/Qwen2.5-Coder-7B-Instruct",          # leaderboard rank 49 (in the data package)
    "ByteDance-Seed/Seed-Coder-8B-Instruct",   # leaderboard rank 44 (in the data package)
]
# T4 alternative (set DTYPE = "half"): "Qwen/Qwen2.5-Coder-3B-Instruct", "Qwen/Qwen2.5-Coder-1.5B-Instruct"

LOOPS = [
    "code_generation_summarization",         # baseline: standard loop
    "code_generation_pseudocode",            # extension: code <-> pseudocode (assert line retained)
    "code_generation_pseudocode_noassert",   # ablation: assert line dropped (optional, remove to save time)
]

MAX_CYCLES        = 10
TEMPERATURE       = 0.0      # greedy decoding
MAX_NEW_TOKENS    = 1024
USE_CHAT_TEMPLATE = True     # format prompts with each model's own chat template
DTYPE             = "auto"   # "half" on T4

DRIVE_DIR    = "/content/drive/MyDrive/A4_evaloop"
RESULTS_DIR  = f"{DRIVE_DIR}/results"
DATA_PACKAGE = f"{DRIVE_DIR}/assignment2-data"   # upload the provided data package folder here

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("data package found:", os.path.exists(f"{DATA_PACKAGE}/leaderboard.json"))

## Install

In [ ]:
%cd /content
!rm -rf EvaLooop && git clone -b {BRANCH} {FORK_URL} EvaLooop
%cd /content/EvaLooop
!pip install -q -e . "vllm==0.11.0" scipy
!git log --oneline -3

In [ ]:
# Download MBPP+ and cache the ground-truth outputs once (canonical solutions only)
!python -c "from evaloop.evaluation.py_testing import load_mbpp_dataset as l; p,_=l(); print(len(p), 'MBPP+ tasks')"

## Smoke test (5 tasks, 2 cycles)
Check the prompts/outputs look sane before spending GPU time on full runs.

In [ ]:
import glob
import json
import shlex
import subprocess

def evaluate(model, loop, out_dir, name, extra=""):
    cmd = (f"evaloop evaluate --model {shlex.quote(model)} --loop_type {loop} "
           f"--max_cycles {MAX_CYCLES} --temperature {TEMPERATURE} --max_new_tokens {MAX_NEW_TOKENS} "
           f"--use_chat_template {USE_CHAT_TEMPLATE} --dtype {DTYPE} "
           f"--auto_batch_size False --preferred_batch_size 512 "
           f"--output_dir {shlex.quote(out_dir)} --experiment_name {name} --log_level WARNING {extra}")
    print(cmd)
    log = f"{out_dir}/{name}_{loop}.log"
    with open(log, "w") as f:
        rc = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT).returncode
    if rc != 0:
        print(open(log).read()[-3000:])
        raise RuntimeError(f"run failed (rc={rc}), see {log}")

smoke_dir = "/content/smoke"
os.makedirs(smoke_dir, exist_ok=True)
evaluate(MODELS[0], "code_generation_pseudocode", smoke_dir, "smoke", "--n_tasks 5 --max_cycles 2")
res = json.load(open(glob.glob(f"{smoke_dir}/smoke_code_generation_pseudocode_results.json")[0]))[0]
for p in res["prompt_results"][:2]:
    print("=" * 80, p["task_id"], "sustained", p["successful_cycles"])
    for c in p["cycles"]:
        print(f"--- cycle {c['cycle']} success={c.get('success')}\n[code]\n{c.get('task_1_output')}\n[next prompt]\n{c.get('task_2_output')}")

## Full runs
One `evaloop evaluate` process per (model, loop), so vLLM releases GPU memory between runs.
Already finished runs (results file on Drive) are skipped.

In [ ]:
import time
def run_name(model):
    return model.split("/")[-1].lower().replace(".", "_").replace("-", "_")

for model in MODELS:
    for loop in LOOPS:
        name = run_name(model)
        target = f"{RESULTS_DIR}/{name}_{loop}_results.json"
        if os.path.exists(target):
            print("skip (done):", target)
            continue
        t0 = time.time()
        evaluate(model, loop, RESULTS_DIR, name)
        print(f"done {model} / {loop} in {(time.time() - t0) / 60:.1f} min")

## Analysis

In [ ]:
ANALYSIS_DIR = f"{DRIVE_DIR}/analysis"
dp = f"--data_package {shlex.quote(DATA_PACKAGE)}" if os.path.exists(f"{DATA_PACKAGE}/leaderboard.json") else ""
!python a4/analyze_a4.py --results_dir {RESULTS_DIR} --out_dir {ANALYSIS_DIR} {dp}

In [ ]:
import pandas as pd
from IPython.display import Image, display
for f in ["summary.csv", "metrics_full.csv", "paired_comparison.csv", "ranking.csv",
          "pseudocode_leakage.csv", "failure_label_counts.csv", "datapackage_check.csv"]:
    path = f"{ANALYSIS_DIR}/{f}"
    if os.path.exists(path):
        print("==", f)
        display(pd.read_csv(path))
display(Image(f"{ANALYSIS_DIR}/survival_curves.png"))

In [ ]:
# Cross-check with the framework's own analyzer (judge-free metrics)
for path in sorted(glob.glob(f"{RESULTS_DIR}/*_results.json")):
    out = f"{ANALYSIS_DIR}/evaloop_analyze/{os.path.basename(path)[:-len('_results.json')]}"
    !evaloop analyze --results_path {shlex.quote(path)} --metrics "ASL_pow,ASL_bias,pass@1,pass_drop" --generate_plots False --output_dir {shlex.quote(out)} --log_level WARNING
    print(os.path.basename(path))
    print(open(f"{out}/analysis_summary.txt").read()[:600])

## Manual inspection
Open `analysis/manual_inspection.md` (Drive) and label every sampled failed loop in
`analysis/manual_labels.csv` (`manual_label`, `notes`). Then re-run the cell below for the label table.

In [ ]:
labels = pd.read_csv(f"{ANALYSIS_DIR}/manual_labels.csv")
print(len(labels), "sampled failed loops,", labels["manual_label"].notna().sum(), "labelled")
display(pd.crosstab([labels["configuration"]], labels["manual_label"].fillna("(unlabelled)"), margins=True))

## Package the submission artefacts

In [ ]:
SUB = f"{DRIVE_DIR}/submission"
!mkdir -p {SUB}/results {SUB}/analysis
!cp {RESULTS_DIR}/*_results.json {SUB}/results/
!cp {ANALYSIS_DIR}/summary.csv {SUB}/
!cp {ANALYSIS_DIR}/*.csv {ANALYSIS_DIR}/*.md {ANALYSIS_DIR}/*.png {SUB}/analysis/
# Patch of all changes against the upstream commit this branch started from
!git remote add upstream https://github.com/evalooop/EvaLooop.git 2>/dev/null; git fetch -q upstream dev
base = subprocess.check_output("git merge-base upstream/dev HEAD", shell=True, text=True).strip()
subprocess.run(f"git diff {base} HEAD > {shlex.quote(SUB)}/a4-pseudocode-loop.patch", shell=True, check=True)
print("patch against upstream", base[:8], ":", sum(1 for _ in open(f"{SUB}/a4-pseudocode-loop.patch")), "lines")
!ls -R {SUB} | head -40